In [1]:
%pwd

'c:\\PROJECTS\\End_to_end_Kidney_Disease_Classification_Deep_learning_project\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\PROJECTS\\End_to_end_Kidney_Disease_Classification_Deep_learning_project'

In [5]:
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/vidyavinodh1982/End_to_end_Kidney_Disease_Classification_Deep_learning_project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="vidyavinodh1982"
os.environ["MLFLOW_TRACKING_PASSWORD"]="9692187617ef13ca72afffa0b0b432392052aca3"

In [6]:
import tensorflow as tf

In [7]:
model =tf.keras.models.load_model("artifacts/training/model.h5")

In [20]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri:str
    params_image_size:list
    params_batch_size: int
    params_is_augmentation:bool


In [21]:
from cnnClassifier.constants import*
from cnnClassifier.utils.common import read_yaml,create_directories,save_json

In [22]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self)-> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model = "artifacts/training/model.h5",
            training_data ="artifacts/data_ingestion/DATASET_KIDNEY",
            mlflow_uri = "https://dagshub.com/vidyavinodh1982/End_to_end_Kidney_Disease_Classification_Deep_learning_project.mlflow",
            all_params = self.params,
            params_image_size = self.params.IMAGE_SIZE,
            params_batch_size = self.params.BATCH_SIZE,
            params_is_augmentation= self.params.AUGMENTATION
        )
        return eval_config

In [23]:
import tensorflow as tf
from pathlib import Path
import mlflow
import mlflow.keras
from urllib.parse import urlparse

In [ ]:
class Evaluation:
    def __init__(self,config:EvaluationConfig):
        self.config = config

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        # validation generator
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            class_mode="categorical", 
            **dataflow_kwargs
        )

        # training generator
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            class_mode="categorical",   
            **dataflow_kwargs
        )
    @staticmethod
    def load_model(path:Path)-> tf.keras.Model:
        return tf.keras.models.load_model(path) 
    
    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self.train_valid_generator()
        self.score = self.model.evaluate(self.valid_generator)
        self.save_score()

    def save_score(self):
        scores = {"loss": self.score[0],"accuracy": self.score[1]}
        save_json(path=Path("scores.json"),data=scores)

    def log_into_mlflow(self):
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        with mlflow.start_run():
        

        # log hyperparameters
        mlflow.log_params(self.config.all_params)

        # log metrics
        mlflow.log_metrics({
            "loss": self.score[0],
            "accuracy": self.score[1]
        })

        # log model
        mlflow.keras.log_model(
            self.model,
            artifact_path="model"
        )

    print("MLflow logging completed successfully")

In [28]:

try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation=Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()
except Exception as e:
        raise e

Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
92/92 ━━━━━━━━━━━━━━━━━━━━ 191s 2s/step - accuracy: 0.7593 - loss: 2.2322


2026/05/18 17:21:17 WARNING mlflow.keras.save: You are saving a Keras model without specifying model signature.
2026/05/18 17:21:38 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\sadha\AppData\Local\Temp\tmp33_6ea0j\model, flavor: keras). Fall back to return ['keras==3.14.0']. Set logging level to DEBUG to see the full traceback. 
2026/05/18 17:21:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'VGG16Model'.
2026/05/18 17:21:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.
2026/05/18 17:21:59 INFO mlflow.tracking._tracking_service.client: 🏃 View run bedecked-mole-524 at: https://dagshub.com/vidyavinodh1982/End_to_e